[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/14_Ontology.ipynb)

#  本体生成(Ontology Generation)

欢迎阅读 Semantica 本体模块的综合指南。该模块是把你的数据组织成有意义的知识图谱的核心引擎,提供了从原始数据到经过校验的 OWL 本体的完整 6 阶段管线。

在本 Notebook 中,我们将深入探讨:
1.  **6 阶段生成管线**:理解 Semantica 如何把数据转化为知识。
2.  **重点组件**:详解 `ClassInferrer`、`PropertyGenerator` 和 `OntologyOptimizer` 的用法。
3.  **可视化**:通过交互式图表和层级结构探索你的本体。
4.  **高级用法**:文本到本体(LLM)、能力问题(Competency Questions)和生命周期管理。
5.  **导出与互操作性**:以 Turtle 和 RDF/XML 等标准格式保存你的成果。

**文档**:[API 参考](https://semantica.readthedocs.io/reference/ontology/)

## 开始使用

首先,让我们设置环境并初始化 `OntologyEngine`。该引擎是所有本体操作的统一入口。

In [ ]:
!pip install -q semantica

In [ ]:

from semantica.ontology import OntologyEngine, OntologyGenerator
from semantica.utils.logging import get_logger

# Initialize logger for visibility
logger = get_logger("ontology_guide")

# Initialize the Engine
# base_uri defines the namespace root for your ontology
engine = OntologyEngine(base_uri="https://docs.semantica.dev/ontology/", min_occurrences=1)

print("Ontology Engine initialized successfully!")

---

## 5 阶段生成管线

Semantica 使用一套精密的 6 阶段管线来稳健地生成本体。这一自动化过程接收原始的实体和关系数据,并产出高质量的 OWL 本体。

### 各个阶段:
1.  **语义网络解析**:从你的输入中抽取原始概念和连接。
2.  **YAML 到定义**:将概念转化为结构化的类定义。
3.  **定义到类型**:把定义映射到形式化的 OWL 类型(例如 `owl:Class`、`owl:ObjectProperty`)。
4.  **层级生成**:使用 `associatedWith` 或语言学模式构建分类结构(父子关系)。
5.  **TTL 生成**:把内存中的结构序列化为 Turtle 格式的逻辑。

让我们用一些示例数据来实际看看。

In [ ]:
# Sample Data: A simple corporate structure
entities = [
    {"id": "e1", "type": "Company", "name": "TechCorp", "founded": "2010"},
    {"id": "e2", "type": "Person", "name": "Alice", "role": "CEO"},
    {"id": "e3", "type": "Person", "name": "Bob", "role": "CTO"},
    {"id": "e4", "type": "Department", "name": "Engineering"},
    {"id": "e5", "type": "Project", "name": "Project Phoenix"},
    # Added entities to ensure robust class inference
    {"id": "e6", "type": "Company", "name": "TechSolutions", "founded": "2015"},
    {"id": "e7", "type": "Person", "name": "Charlie", "role": "Lead"},
    {"id": "e8", "type": "Person", "name": "Dave", "role": "Manager"},
    {"id": "e9", "type": "Department", "name": "Research"},
    {"id": "e10", "type": "Project", "name": "Project Orion"}
]

relationships = [
    {"source": "e2", "target": "e1", "type": "leads"},
    {"source": "e3", "target": "e4", "type": "manages"},
    {"source": "e4", "target": "e1", "type": "part_of"},
    {"source": "e3", "target": "e5", "type": "works_on"},
    # Additional relationships for new entities
    {"source": "e7", "target": "e6", "type": "leads"},
    {"source": "e8", "target": "e9", "type": "manages"},
    {"source": "e9", "target": "e6", "type": "part_of"},
    {"source": "e7", "target": "e10", "type": "works_on"}
]

data = {
    "entities": entities,
    "relationships": relationships
}

# Run the full pipeline
ontology = engine.from_data(data, name="CorporateOntology")

print(f"Generated Ontology: {ontology['name']}")
print(f"Classes Found: {len(ontology['classes'])}")
print(f"Properties Found: {len(ontology['properties'])}")

### 检查结果

生成的 `ontology` 对象是一个包含所有推断结构的丰富字典。让我们窥探一下内部,看看创建了哪些类和属性。

In [ ]:
# Inspect Classes
print("--- Inferred Classes ---")
for cls in ontology['classes']:
    print(f"Class: {cls['name']}")
    print(f"  URI: {cls.get('uri')}")
    # Check if a hierarchy was inferred
    if cls.get('subClassOf'):
        print(f"  Parent: {cls['subClassOf']}")
    print("")

# Inspect Properties
print("--- Inferred Properties ---")
for prop in ontology['properties']:
    type_label = "Object Property" if prop['type'] == 'object' else "Data Property"
    print(f"{prop['name']} [{type_label}]")
    print(f"  Domain: {prop.get('domain')}")
    print(f"  Range: {prop.get('range')}")
    print("")

---

## 深入探讨:逐个组件

虽然 `OntologyEngine` 非常适合一次性生成,但你通常需要更精细的控制。让我们看看驱动该引擎的各个工具。

### 1. `ClassInferrer`:掌握类的发现

`ClassInferrer` 通过分析实体来发现模式。它能够处理噪声,并且只会为出现频率足够高的类型创建类。

*   **`min_occurrences`**:忽略实体数量少于该值的类型。
*   **`build_class_hierarchy`**:切换是否自动检测父子关系。

In [ ]:
from semantica.ontology import ClassInferrer

# Initialize inferrer with a threshold
# We set min_occurrences=1 here to capture everything in our small example
inferrer = ClassInferrer(min_occurrences=1)

raw_entities = [
    {"type": "Manager", "name": "Dave", "level": 5},
    {"type": "Manager", "name": "Eve", "level": 4},
    {"type": "Employee", "name": "Frank"}, # Only 1 employee
    {"type": "TemporaryWorker", "name": "Grace"} 
]

# Infer classes
classes = inferrer.infer_classes(raw_entities, build_hierarchy=True)

print(f"Inferred {len(classes)} classes from raw entities.")
for c in classes:
    print(f"- {c['name']} (Count: {c['entity_count']})")

### 2. `PropertyGenerator`:本体的粘合剂

属性定义关系。Semantica 区分以下两种属性:
*   **对象属性(Object Properties)**:两个实体之间的链接(例如 Person 与 Company 之间的 `leads`)。
*   **数据属性(Data Properties)**:实体的特性(例如 Company 的 `founded` 年份)。

`PropertyGenerator` 会自动检测这一区别。

In [ ]:
from semantica.ontology import PropertyGenerator

prop_gen = PropertyGenerator()

# We need the classes first to help property generation context
context_classes = classes  # reusing from previous step

# Let's define some relationships and attributes implicitly via entities
# Note: 'level' in Manager entities is a potential data property
complex_entities = [
    {"id": "m1", "type": "Manager", "name": "Dave", "level": 5},
    {"id": "e1", "type": "Employee", "name": "Frank"}
]
complex_relationships = [
    {"source": "m1", "target": "e1", "type": "supervises"} # Object property
]

properties = prop_gen.infer_properties(
    entities=complex_entities,
    relationships=complex_relationships,
    classes=context_classes
)

print("--- Property Types Identified ---")
for p in properties:
    print(f"Property: {p['name']}")
    print(f"  Type: {p['type']}")
    print(f"  Domain: {p['domain']} -> Range: {p['range']}")

### 3. `OntologyOptimizer`:精炼结构

在定稿之前,最好先做优化。优化器会移除冗余并提升一致性,例如确保所有类都有恰当的标签和有效的 URI。

In [ ]:
from semantica.ontology import OntologyOptimizer

optimizer = OntologyOptimizer()

# Let's pretend we have a messy ontology dict
messy_ontology = {
    "classes": [
        {"name": "Person", "uri": "...Person"},
        {"name": "Person", "uri": "...Person"} # Duplicate!
    ],
    "properties": []
}

clean_ontology = optimizer.optimize_ontology(messy_ontology, remove_redundancy=True)

print(f"Original Classes: {len(messy_ontology['classes'])}")
print(f"Optimized Classes: {len(clean_ontology['classes'])}")

---

## 可视化

一图胜千言!`OntologyVisualizer` 让你以交互方式探索本体的结构。

我们可以可视化:
*   **类层级**:类继承的树状图。
*   **结构网络**:类与属性的完整图。
*   **指标仪表盘**:一目了然的高层统计。

In [ ]:
from semantica.visualization import OntologyVisualizer

viz = OntologyVisualizer()

# 1. Interactive Class Hierarchy
# Returns a Plotly figure you can interact with
fig_hierarchy = viz.visualize_hierarchy(ontology, output="interactive")
if fig_hierarchy:
    fig_hierarchy.show()

# 2. Ontology Structure Network
# See how classes and properties connect
fig_structure = viz.visualize_structure(ontology, output="interactive")
if fig_structure:
    fig_structure.show()

# 3. Metrics Dashboard
# View counts, depths, and statistics
fig_metrics = viz.visualize_metrics(ontology, output="interactive")
if fig_metrics:
    fig_metrics.show()

---

## 高级用法:生命周期与 AI

企业本体是不断演化的产物。Semantica 提供了管理其整个生命周期的工具,并借助 AI 加速创建。

### 1. 文本到本体(LLM 集成)

无需手动创建实体,可使用 `LLMOntologyGenerator` 直接从文本需求或文档中抽取本体。

In [ ]:
from semantica.ontology import LLMOntologyGenerator

try:
    # Note: Requires an API key in your environment variables
    llm_gen = LLMOntologyGenerator(provider="openai", model="gpt-4")

    text_description = """
    A University has many Departments. Each Department offers several Courses.
    Professors teach Courses and belong to a Department.
    Students enroll in Courses.
    """

    llm_ontology = llm_gen.generate_ontology_from_text(
        text=text_description,
        name="UniversityOntology"
    )

    print("Generated Classes:", [c['name'] for c in llm_ontology['classes']])
except Exception:
    print("Skipping LLM generation: No API key or provider configured in this environment.")

### 2. 测试驱动设计(能力问题)

将你的需求形式化为"能力问题"(Competency Questions,CQ)。`CompetencyQuestionsManager` 可以检查你的本体是否包含回答它们所需的术语。

In [ ]:
from semantica.ontology import CompetencyQuestionsManager

cq_manager = CompetencyQuestionsManager()

# Define what our ontology SHOULD answer
cq_manager.add_question("Who leads TechCorp?", category="organizational")
cq_manager.add_question("Which projects does Bob manage?", category="operational")

# Validate our 'ontology' against these questions
validation_results = cq_manager.validate_ontology(ontology)

print(f"Answerable Questions: {validation_results['answerable']} / {validation_results['total_questions']}")
for q in cq_manager.questions:
    status = "✅" if q.answerable else "❌"
    print(f"{status} {q.question}")

### 3. 生命周期管理(版本与复用)

使用 `VersionManager` 管理迭代,并使用 `ReuseManager` 导入 FOAF 或 Dublin Core 等外部标准。

In [ ]:
from semantica.change_management import VersionManager
from semantica.ontology import ReuseManager

# --- Versioning ---
v_manager = VersionManager(base_uri="https://example.org/ontology/")
v1 = v_manager.create_version("1.0", ontology, changes=["Initial creation"])
print(f"Created Version: {v1.version} at {v1.ontology_iri}")

# --- Reuse ---
reuse_manager = ReuseManager()

# Check if we can reuse FOAF
foaf_info = reuse_manager.research_ontology("http://xmlns.com/foaf/0.1/")
if foaf_info:
    print(f"Found standard ontology: {foaf_info['name']}")
    # We could now import this into our ontology
    ontology['imports'].append(foaf_info['uri'])

---

## 导出你的本体

在本体构建并校验完成之后,你会想要保存它。Semantica 以 **Turtle(`.ttl`)** 作为主要格式,同时通过 `rdflib` 支持其他格式。

你可以导出为字符串,也可以直接导出到文件。

In [ ]:
# Get Turtle string representation
ttl_output = engine.to_owl(ontology, format="turtle")

print("--- Turtle Preview (First 500 chars) ---")
print(ttl_output[:500])
print("...")

# Save to file
output_path = "corporate_ontology.ttl"
engine.export_owl(ontology, path=output_path, format="turtle")
print(f"Successfully saved ontology to {output_path}")

## 总结

你现在已经掌握了 Semantica 本体模块的要点!

*   **自动化生成**:使用 6 阶段管线,从原始数据得到结构化的本体。
*   **组件控制**:使用 `ClassInferrer` 和 `PropertyGenerator` 进行精细建模。
*   **可视化**:以交互方式探索了本体结构。
*   **高级生命周期**:使用了 AI 生成、能力问题和版本管理。
*   **导出**:把你的知识图谱序列化,以便在其他语义网工具中使用。

**下一步**:
*   尝试自定义 `NamespaceManager`,使用你所在组织的 URL。
*   探索 `OntologyEvaluator`,获取更深入的质量指标。
*   把生成的本体输入**知识图谱**模块,开始对你的数据进行推理!
*   在[语义层基础](./26_Semantic_Layer_Basics.ipynb)中,把图、本体和显式映射整合在一起。